In [2]:
import pandas as pd
import numpy as np
import sys
import os
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

sys.path.append(os.path.abspath('../'))
from src.modeling_pipeline import evaluate_model_cv

# Load final processed files from Task 1
fraud_df = pd.read_csv('../data/processed/fraud_features_final.csv')
cc_df = pd.read_csv('../data/processed/creditcard_cleaned.csv')

In [3]:
X_fraud = fraud_df.drop(columns=['class'])
y_fraud = fraud_df['class']

results_fraud = []

print("--- Running E-Commerce Data Models ---")
# Baseline model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
results_fraud.append(evaluate_model_cv(X_fraud, y_fraud, lr_model, "Logistic Regression (Baseline)", apply_smote=True))

# Tuned Ensemble model
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
results_fraud.append(evaluate_model_cv(X_fraud, y_fraud, xgb_model, "XGBoost Ensemble", apply_smote=True))

# Build Performance Comparison Frame
df_fraud_perf = pd.DataFrame(results_fraud).drop(columns=['confusion_matrix', 'fitted_model'])
print(df_fraud_perf)

# Save the best model artifact
joblib.dump(results_fraud[1]['fitted_model'], '../models/best_ecommerce_xgb.pkl')

--- Running E-Commerce Data Models ---


2026-06-15 23:27:35,205 - INFO - [Logistic Regression (Baseline)] CV AUC-PR: 0.6626 ± 0.0062 | Mean F1: 0.6752 ± 0.0046
2026-06-15 23:27:38,289 - INFO - [XGBoost Ensemble] CV AUC-PR: 0.7192 ± 0.0062 | Mean F1: 0.6984 ± 0.0051


                       model_name  mean_auc_pr  std_auc_pr   mean_f1    std_f1
0  Logistic Regression (Baseline)     0.662568    0.006212  0.675199  0.004599
1                XGBoost Ensemble     0.719215    0.006181  0.698378  0.005115


['../models/best_ecommerce_xgb.pkl']

In [6]:
# --- Running Bank Credit Card Models (Optimized) ---
from sklearn.preprocessing import StandardScaler

X_cc = cc_df.drop(columns=['Class'])
y_cc = cc_df['Class']

# FIX 1: Robustly scale the 'Amount' and 'Time' columns so Logistic Regression converges quickly
columns_to_scale = ['Time', 'Amount']
X_cc_scaled = X_cc.copy()
scaler = StandardScaler()
X_cc_scaled[columns_to_scale] = scaler.fit_transform(X_cc[columns_to_scale])

results_cc = []

print("\n--- Running Bank Credit Card Models ---")

# FIX 2: Increase max_iter and change solver to 'saga' or 'liblinear' which handles large datasets better,
# or simply stick to lbfgs now that the features are scaled perfectly!
lr_cc = LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42, n_jobs=-1)
results_cc.append(evaluate_model_cv(X_cc_scaled, y_cc, lr_cc, "Logistic Regression (Baseline)", apply_smote=True))

# Random Forest Ensemble (This handles unscaled/scaled data identically, but likes n_jobs=-1 for speed)
rf_cc = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
results_cc.append(evaluate_model_cv(X_cc_scaled, y_cc, rf_cc, "Random Forest Ensemble", apply_smote=True))

# Build Performance Comparison Frame
df_cc_perf = pd.DataFrame(results_cc).drop(columns=['confusion_matrix', 'fitted_model'])
print(df_cc_perf)


--- Running Bank Credit Card Models ---


c:\Users\abbadal\Desktop\TENX\WEEK5\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\abbadal\Desktop\TENX\WEEK5\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\abbadal\Desktop\TENX\WEEK5\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\abbadal\Desktop\TENX\WEEK5\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.

                       model_name  mean_auc_pr  std_auc_pr   mean_f1    std_f1
0  Logistic Regression (Baseline)     0.733392    0.021674  0.268321  0.009048
1          Random Forest Ensemble     0.776057    0.042154  0.734228  0.010378
